In [1]:
import torchvision.models as visionmodels
import torch.nn as nn
import torch.optim as optim
import torch

class Emolens(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = visionmodels.efficientnet_b0(weights=visionmodels.EfficientNet_B0_Weights.DEFAULT)

        for parameter in self.model.parameters():
            parameter.requires_grad = False

        for parameter in self.model.features[-3].parameters():
            parameter.requires_grad = True

        self.model.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, 7))

    def forward(self, x):
        x = self.model(x)

        return x

In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('mv kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
os.system('kaggle datasets download -d msambare/fer2013 -p /content/ --unzip')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


0

In [4]:
train_Data = '/content/train'
Test_Data = '/content/test'
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_dataset = datasets.ImageFolder(root=train_Data, transform=train_transform)
test_dataset = datasets.ImageFolder(root=Test_Data, transform=test_transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [5]:
model = Emolens()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.CrossEntropyLoss()
model_parameters = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(model_parameters, lr=0.0001)

In [ ]:
num_of_epochs=10

for epoch in range(num_of_epochs):
    sum_of_train_losses = 0
    sum_of_test_losses = 0
    total_correct = 0
    total_images = 0

    model.train()
    for images, labels in train_dataloader:
        images = images.to(device)
        labels = labels.to(device)
        
        y_pred = model(images)
        loss = criterion(y_pred, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_of_train_losses += loss.item()

    model.eval()
    with torch.no_grad():
        for images, labels in test_dataloader:
            images = images.to(device)
            labels = labels.to(device)
            
            y_pred = model(images)
            loss = criterion(y_pred, labels)

            y_pred = torch.argmax(y_pred, dim=1)
            total_correct += (y_pred == labels).sum().item()
            total_images += labels.size(0)

            sum_of_test_losses += loss.item()

    accuracy = (total_correct / total_images) * 100
    print(f'Epoch {epoch+1}: Train Loss: {sum_of_train_losses / len(train_dataloader): .3f} | Test Loss: {sum_of_test_losses / len(test_dataloader): .3f} | Accuracy: {accuracy: .2f}%')

Epoch 1: Train Loss:  1.426 | Test Loss:  1.149 | Accuracy:  56.12%
